In [1]:
!pip install -qU  langchain-community langchain-openai

In [2]:
import langchain
langchain.__version__

'1.2.15'

## SQLDatabase Toolkit

This will help you get started with the SQL Database toolkit. For detailed documentation of all SQLDatabaseToolkit features and configurations head to the API reference.

Tools within the SQLDatabaseToolkit are designed to interact with a SQL database.

A common application is to enable agents to answer questions using data in a relational database, potentially in an iterative fashion (e.g., recovering from errors).

### Instantiation
The **`SQLDatabaseToolkit`** toolkit requires:
*   a **SQLDatabase** object;
*   a LLM or chat model (for instantiating the **QuerySQLCheckerTool** tool).

Below, we instantiate the toolkit with these objects. Let’s first create a database object.

This guide uses the example **`Chinook`** database based on these instructions.

Below we will use the requests library to pull the .sql file and create an in-memory SQLite database. Note that this approach is lightweight, but ephemeral and not thread-safe. If you’d prefer, you can follow the instructions to save the file locally as `**Chinook.db**` and instantiate the database via **`db = SQLDatabase.from_uri("sqlite:///Chinook.db")`**.

In [3]:
import sqlite3

import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


engine = get_engine_for_chinook_db()

db = SQLDatabase(engine)

In [5]:
import os
from google.colab import userdata

if userdata.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [6]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
    )

### Tools

In [7]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [8]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7ae4c7ac5310>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7ae4c7ac5310>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7ae4c7ac5310>),
 QuerySQLCheckerTool(description='Use this tool to double check

In [9]:
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
    QuerySQLDatabaseTool,
)

### Use within an agent

In [10]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

assert len(prompt_template.messages) == 1
print(prompt_template.input_variables)

['dialect', 'top_k']


In [11]:
prompt_template

ChatPromptTemplate(input_variables=['dialect', 'top_k'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': 'langchain-ai', 'lc_hub_repo': 'sql-agent-system-prompt', 'lc_hub_commit_hash': '31156d5fe3945188ee172151b086712d22b8c70f8f1c0505f5457594424ed352'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['dialect', 'top_k'], input_types={}, partial_variables={}, template='You are an agent designed to interact with a SQL database.\nGiven an input question, create a syntactically correct {dialect} query to run, then look at the results of the query and return the answer.\nUnless the user specifies a specific number of examples they wish to obtain, always limit your query to at most {top_k} results.\nYou can order the results by a relevant column to return the most interesting examples in the database.\nNever query for all the columns from a specific table, only ask for the relevant columns given the question.\nYou have access to tools for interact

In [12]:
system_message = prompt_template.format(dialect="SQLite", top_k=5)

In [13]:
system_message

'System: You are an agent designed to interact with a SQL database.\nGiven an input question, create a syntactically correct SQLite query to run, then look at the results of the query and return the answer.\nUnless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 5 results.\nYou can order the results by a relevant column to return the most interesting examples in the database.\nNever query for all the columns from a specific table, only ask for the relevant columns given the question.\nYou have access to tools for interacting with the database.\nOnly use the below tools. Only use the information returned by the below tools to construct your final answer.\nYou MUST double check your query before executing it. If you get an error while executing a query, rewrite the query and try again.\n\nDO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database.\n\nTo start you should ALWAYS look at the tables in the datab

In [17]:
from langchain.agents import create_agent

agent = create_agent(llm, toolkit.get_tools(), system_prompt=system_message)

In [18]:
example_query = "Which country's customers spent the most?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Which country's customers spent the most?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_pY2T1UxqCvGxz3fMCyJWBkxA)
 Call ID: call_pY2T1UxqCvGxz3fMCyJWBkxA
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_Rfqdm11ygEyxVrlyjmkh3yNL)
 Call ID: call_Rfqdm11ygEyxVrlyjmkh3yNL
  Args:
    table_names: Customer
  sql_db_schema (call_GoQlO5YWunWzkbDfKK8wEJTn)
 Call ID: call_GoQlO5YWunWzkbDfKK8wEJTn
  Args:
    table_names: Invoice
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Invoice" (
	"Inv

In [19]:
example_query = "Who are the top 3 best selling artists?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who are the top 3 best selling artists?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_RsaVC3Evq14kOPW0NCDZvk0E)
 Call ID: call_RsaVC3Evq14kOPW0NCDZvk0E
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_FDAfbuKSCNEjLs0hiIxBfYym)
 Call ID: call_FDAfbuKSCNEjLs0hiIxBfYym
  Args:
    table_names: Artist
  sql_db_schema (call_yRI7iL2oMykr7Pk8jGTkjZLj)
 Call ID: call_yRI7iL2oMykr7Pk8jGTkjZLj
  Args:
    table_names: InvoiceLine
  sql_db_schema (call_EoUV4MEhPtNXZ6RQPoC56Ctv)
 Call ID: call_EoUV4MEhPtNXZ6RQPoC56Ctv
  Args:
    table_names: Track
==============

In [20]:
example_query = "Who are the top 10 best selling artists?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who are the top 10 best selling artists?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_pjVRdO1GLZ47txsT5K6ARumt)
 Call ID: call_pjVRdO1GLZ47txsT5K6ARumt
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_qRFk5uAqcPZC0HkYiJeDyH9p)
 Call ID: call_qRFk5uAqcPZC0HkYiJeDyH9p
  Args:
    table_names: Artist
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)

/*
3 rows from Artist table:
ArtistId	Name
1	A